# What card is actually in your neobank app?

Almost every neobank ships a card, and the card is the part of the product that is
hardest to fake: it needs a network, an issuer and a settlement path. So the card
fields are a decent proxy for how real, and how independent, a product is.

This notebook looks at the network duopoly, where domestic networks win instead, what
kinds of card exist, and the products that ship no card at all. `card_network` and
`card_type` are populated for 308 of 368 rows — that gap is itself a finding, covered
in section 4.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Kaggle mounts inputs in two different places depending on how the dataset was
# attached: /kaggle/input/<slug>/ from the UI, /kaggle/input/datasets/<owner>/<slug>/
# when declared through the API. Searching for the file covers both, and keeps this
# working if you fork the notebook and attach the data yourself.
def find_entities():
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for hit in sorted(kaggle_input.rglob("entities.csv")):
            return hit
    for local in (Path("entities.csv"), Path("../../.staging/entities.csv")):
        if local.exists():
            return local
    return None


src = find_entities()
if src is None:
    raise FileNotFoundError(
        "entities.csv not found. On Kaggle, add the neobankbeat/neobanks dataset via "
        "'Add Input'. Locally, download it from "
        "https://www.kaggle.com/datasets/neobankbeat/neobanks"
    )
df = pd.read_csv(src)

plt.rcParams.update({"figure.figsize": (9, 5), "axes.spines.top": False, "axes.spines.right": False})
print(f"{len(df)} neobanks · {df.shape[1]} columns")
df.head(3)

## 1. Visa and Mastercard, and not much else

`card_network` is recorded verbatim, so it needs light normalising: some products issue
on both networks, and some on a domestic network alongside a global one.

In [ ]:
def network_group(value):
    if pd.isna(value):
        return "no card recorded"
    v = str(value)
    has_visa, has_mc = "Visa" in v, "MC" in v or "Mastercard" in v
    # A domestic scheme named alongside Visa/MC still means local rails are involved.
    domestic = any(tag in v for tag in ("RuPay", "Verve", "Mada", "dom.", "Elo", "UnionPay", "Troy", "Mir"))
    if domestic:
        return "domestic scheme"
    if has_visa and has_mc:
        return "Visa + Mastercard"
    if has_visa:
        return "Visa"
    if has_mc:
        return "Mastercard"
    return "other"


groups = df["card_network"].map(network_group).value_counts()
ax = groups.sort_values().plot.barh(color="#1d4ed8")
ax.set(title="Card network behind each neobank", xlabel="neobanks", ylabel="")
plt.tight_layout()
plt.show()

carded = df["card_network"].notna().sum()
print(f"{carded} of {len(df)} products have a card network recorded ({100 * carded / len(df):.0f}%)")
groups.to_frame("neobanks")

## 2. Domestic schemes are an emerging-market story

RuPay in India, Verve in Nigeria, Mada in Saudi Arabia. Where a domestic scheme shows
up, it is usually because local acceptance or regulation makes it the default rail
rather than a choice.

In [ ]:
dom = df[df["card_network"].map(network_group).eq("domestic scheme")]
print(f"{len(dom)} products issue on a domestic scheme\n")
print(dom[["name", "hq", "card_network", "regulation_type"]].to_string(index=False))

by_region = dom["region"].value_counts()
ax = by_region.sort_values().plot.barh(color="#b45309")
ax.set(title="Domestic-scheme cards by region", xlabel="neobanks", ylabel="")
plt.tight_layout()
plt.show()

## 3. Debit dominates, credit barely exists

`card_type` is free text with 55 distinct spellings, so this groups by the capabilities
mentioned rather than trusting the exact string. One product can appear in several rows
here — a card that is both debit and credit is counted in both.

In [ ]:
KINDS = {
    "debit": ["debit"],
    "credit": ["credit"],
    "prepaid": ["prepaid"],
    "virtual": ["virtual"],
    "business": ["business", "corporate"],
    "crypto-settled": ["crypto"],
    "wallet-only": ["wallet"],
}

types = df["card_type"].dropna().str.lower()
counts = {kind: int(types.str.contains("|".join(words)).sum()) for kind, words in KINDS.items()}
counts = pd.Series(counts).sort_values()

ax = counts.plot.barh(color="#0f766e")
ax.set(title="Card capabilities mentioned (products can appear in more than one)", xlabel="neobanks", ylabel="")
plt.tight_layout()
plt.show()

print(f"of {len(types)} products with a card type recorded")
counts.sort_values(ascending=False).to_frame("neobanks")

## 4. The 60 with no card recorded

This is the gap worth reading, not skipping. Some of these genuinely have no card — a
self-custodial wallet is an interface, not an issuer. For others the card simply is not
verified yet. Splitting by `custody` separates the two cases.

In [ ]:
no_card = df[df["card_network"].isna()]
print(f"{len(no_card)} products have no card network recorded\n")

split = pd.crosstab(no_card["custody"], no_card["category"])
print(split.to_string())

self_cust = no_card["custody"].str.contains("Self-custodial|MPC", na=False).sum()
print(f"\n{self_cust} of the {len(no_card)} are self-custodial — for these, no card is the product design.")
print(f"{len(no_card) - self_cust} are custodial, where a card may exist but is unverified.")

## 5. Cards that settle in crypto, and cards without ID

Two segments that only exist because the card layer is rented rather than owned: cards
funded from stablecoins, and cards issued against a wallet that never asked who you are.

In [ ]:
stable = df["stablecoins"].fillna(False).astype(bool)
kyc = df["kyc"].fillna("Unknown")

print(f"support stablecoins        : {stable.sum():>3}  ({100 * stable.mean():.0f}%)")
print(f"no KYC at all             : {kyc.eq('No').sum():>3}")
print(f"KYC only for the card     : {kyc.eq('Card only').sum():>3}")
print()

overlap = pd.crosstab(kyc, stable.map({True: "stablecoins", False: "no stablecoins"}))
ax = overlap.plot.bar(stacked=True, color=["#94a3b8", "#7c3aed"], figsize=(9, 5))
ax.set(title="KYC posture against stablecoin support", xlabel="KYC required", ylabel="neobanks")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

overlap

## 6. Build a shortlist by card

Filter by the properties that actually differ between products on the same rails.

In [ ]:
NETWORK = "Visa"        # "Mastercard", "Visa + Mastercard", "domestic scheme"
NEEDS_CRYPTO = False    # True to require stablecoin support

mask = df["card_network"].map(network_group).eq(NETWORK)
if NEEDS_CRYPTO:
    mask &= df["stablecoins"].fillna(False).astype(bool)

cols = ["name", "hq", "card_network", "card_type", "custody", "regulation_type", "kyc"]
print(f"{mask.sum()} matches")
df.loc[mask, cols].sort_values("name").reset_index(drop=True).head(25)

## Caveats, and how to cite this

Three things to know before quoting any number above.

**Coverage is deliberate.** Only live, consumer-facing products are tracked. Defunct
neobanks and pure BaaS/infrastructure providers are excluded, so this cannot be used
for survival analysis — the denominator is "what exists now", not "what was ever launched".

**Empty is not zero.** Unverified fields are left blank rather than guessed, so a missing
value means "not confirmed from a primary source", not "does not have it". Treat every
count here as a floor.

**Self-disclosed figures are not audited.** Where user counts, funding or volume appear,
they are what companies chose to announce, on the metric they chose to announce it on.

Data: [neobankbeat/neobanks](https://www.kaggle.com/datasets/neobankbeat/neobanks) ·
methodology and field dictionary: [neobankbeat.com/data/](https://www.neobankbeat.com/data/)

> neobankbeat (2026). *Open directory of neobanks worldwide.* https://www.neobankbeat.com/ (MIT).